# Download CNN/DailyMail articles

This notebook downloads and prepares the external Stage-1 news summarization dataset separately from the OPT training notebook.

This version avoids both Hugging Face datasets-server row requests and Hugging Face streaming clients. It downloads the published CNN/DailyMail Parquet files once, reads them locally, filters the required samples, and creates:

- `data/raw/raw_news_stage1.jsonl`
- `data/processed/stage1_train.jsonl`
- `data/processed/stage1_validation.jsonl`

The OPT-2.7B notebook only reads the two processed Stage-1 JSONL files.


## Import libraries and configure the download

The Stage-1 training records come from CNN/DailyMail's published **train Parquet shards**, while Stage-1 validation records come from its published **validation Parquet file**.

The notebook downloads files with ordinary HTTP requests and then performs all filtering locally.


In [ ]:
import json
import os
import random
import re
import time
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PARQUET_CACHE_DIR = RAW_DIR / "cnn_dailymail_parquet"

RAW_PATH = RAW_DIR / "raw_news_stage1.jsonl"
STAGE1_TRAIN_PATH = PROCESSED_DIR / "stage1_train.jsonl"
STAGE1_VAL_PATH = PROCESSED_DIR / "stage1_validation.jsonl"
STAGE2_SPLIT_PATHS = [
    PROCESSED_DIR / "train.jsonl",
    PROCESSED_DIR / "validation.jsonl",
    PROCESSED_DIR / "test.jsonl",
]

TRAIN_SAMPLE_SIZE = 5000
VAL_SAMPLE_SIZE = 500

MAX_ARTICLE_WORDS_EXCLUSIVE = 500
SEED = 67
OVERWRITE = True

# CHANGED: Direct published Parquet files.
# The notebook starts with train shard 0 and downloads another shard
# only if the first shard does not contain enough qualifying examples.
BASE_URL = "https://huggingface.co/datasets/abisee/cnn_dailymail/resolve/main/3.0.0"


TRAIN_PARQUET_URLS = [
    f"{BASE_URL}/train-00000-of-00003.parquet?download=true",
    f"{BASE_URL}/train-00001-of-00003.parquet?download=true",
    f"{BASE_URL}/train-00002-of-00003.parquet?download=true",
]

VALIDATION_PARQUET_URL = (
    f"{BASE_URL}/validation-00000-of-00001.parquet?download=true"
)

WORD_RE = re.compile(
    r"\b\w+(?:[’'-]\w+)*\b",
    flags=re.UNICODE,
)

RAW_COLUMNS = [
    "article_id",
    "article",
    "summary",
    "article_word_count",
]

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

print(f"Parquet cache: {PARQUET_CACHE_DIR}")
print(f"Raw output: {RAW_PATH}")
print(f"Stage-1 train output: {STAGE1_TRAIN_PATH}")
print(f"Stage-1 validation output: {STAGE1_VAL_PATH}")
print(f"Training samples: {TRAIN_SAMPLE_SIZE}")
print(f"Validation samples: {VAL_SAMPLE_SIZE}")
print(
    f"Article filter: < {MAX_ARTICLE_WORDS_EXCLUSIVE} words"
)
print("Stage-1 records overlapping any Stage-2 split will be skipped and replaced.")


Parquet cache: data\raw\cnn_dailymail_parquet
Raw output: data\raw\raw_news_stage1.jsonl
Stage-1 train output: data\processed\stage1_train.jsonl
Stage-1 validation output: data\processed\stage1_validation.jsonl
Training samples: 5000
Validation samples: 500
Article filter: < 500 words
Stage-1 records overlapping any Stage-2 split will be skipped and replaced.


## Download and filter CNN/DailyMail

This version does **not** use `load_dataset(..., streaming=True)` and does **not** use the datasets-server `/rows` endpoint.

It downloads the published Parquet files with a normal `requests` session configured with retry/backoff. Once a file is downloaded, all sampling and filtering happen locally.


In [2]:
def count_words(text: str) -> int:
    return len(
        WORD_RE.findall(
            str(text)
        )
    )


# ADDED: ordinary HTTP client with retry/backoff.
# This avoids the Hugging Face streaming client's "client has been closed"
# error and is much less request-intensive than the /rows API.
def create_download_session():
    retry = Retry(
        total=8,
        connect=8,
        read=8,
        status=8,
        backoff_factor=2.0,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504,
        ],
        allowed_methods=[
            "GET",
        ],
        respect_retry_after_header=True,
    )

    adapter = HTTPAdapter(max_retries=retry)

    session = requests.Session()
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update(
        {
            "User-Agent": (
                "Mozilla/5.0 "
                "(Stage1-News-Summary-Dataset)"
            )
        }
    )

    return session


def download_file(url: str, destination: Path):
    """
    Download one Parquet file once.
    Existing non-empty files are reused so interrupted notebook runs
    do not redownload completed files.
    """
    if (destination.exists() and destination.stat().st_size > 0):
        print(f"Using cached file: {destination}")
        return destination

    temporary_path = destination.with_suffix(destination.suffix + ".part")

    if temporary_path.exists():
        temporary_path.unlink()

    print(f"Downloading {destination.name} ...")

    session = create_download_session()

    try:
        with session.get(url, stream=True, timeout=(30, 300), allow_redirects=True) as response:
            response.raise_for_status()

            total_bytes = int(
                response.headers.get(
                    "content-length", 0
                )
            )

            downloaded = 0
            next_report = 50 * 1024 * 1024

            with temporary_path.open("wb") as handle:
                for chunk in response.iter_content(
                    chunk_size=4 * 1024 * 1024
                ):
                    if not chunk:
                        continue

                    handle.write(chunk)
                    downloaded += len(chunk)

                    if (
                        downloaded >= next_report
                    ):
                        if total_bytes > 0:
                            print(
                                f"  {downloaded / 1024**2:.0f} MB / "
                                f"{total_bytes / 1024**2:.0f} MB"
                            )
                        else:
                            print(
                                f"  {downloaded / 1024**2:.0f} MB"
                            )

                        next_report += (
                            50 * 1024 * 1024
                        )

        if (
            not temporary_path.exists()
            or temporary_path.stat().st_size == 0
        ):
            raise RuntimeError(
                f"Downloaded file is empty: {url}"
            )

        os.replace(
            temporary_path,
            destination,
        )

        print(
            f"Saved: {destination} "
            f"({destination.stat().st_size / 1024**2:.1f} MB)"
        )

        return destination

    finally:
        session.close()

        if (
            temporary_path.exists()
            and temporary_path.stat().st_size == 0
        ):
            temporary_path.unlink()


def load_stage2_exclusions(paths):
    """Load Stage-2 IDs and articles so Stage 1 remains independent."""
    excluded_ids = set()
    excluded_articles = set()

    for path in paths:
        if not path.exists():
            raise FileNotFoundError(
                f"Missing Stage-2 split '{path}'. Run split_data.ipynb first."
            )

        with path.open('r', encoding='utf-8') as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue

                record = json.loads(line)
                article_id = str(record.get('article_id', '')).strip()
                article = str(record.get('article', '')).strip()

                if not article_id or not article:
                    raise ValueError(
                        f"Invalid Stage-2 record at {path}:{line_number}."
                    )

                excluded_ids.add(article_id)
                excluded_articles.add(article)

    return excluded_ids, excluded_articles


STAGE2_EXCLUDED_IDS, STAGE2_EXCLUDED_ARTICLES = load_stage2_exclusions(
    STAGE2_SPLIT_PATHS
)

print(f"Loaded {len(STAGE2_EXCLUDED_IDS)} Stage-2 IDs to exclude from Stage 1.")


def records_from_parquet(
    parquet_path: Path,
):
    """
    Read a downloaded Parquet file locally and return only qualifying rows.
    No network access occurs here.
    """
    frame = pd.read_parquet(
        parquet_path,
        columns=[
            "id",
            "article",
            "highlights",
        ],
    )

    qualifying = []
    seen_ids = set()
    seen_articles = set()

    for row in frame.itertuples(
        index=False
    ):
        article_id = str(
            row.id
        ).strip()

        article = str(
            row.article
        ).strip()

        summary = str(
            row.highlights
        ).strip()

        if (
            not article_id
            or not article
            or not summary
        ):
            continue

        if (
            article_id in STAGE2_EXCLUDED_IDS
            or article in STAGE2_EXCLUDED_ARTICLES
            or article_id in seen_ids
            or article in seen_articles
        ):
            continue

        article_word_count = count_words(
            article
        )

        if (
            article_word_count
            >= MAX_ARTICLE_WORDS_EXCLUSIVE
        ):
            continue

        qualifying.append(
            {
                "article_id": article_id,
                "article": article,
                "summary": summary,
                "article_word_count": (
                    article_word_count
                ),
            }
        )
        seen_ids.add(article_id)
        seen_articles.add(article)

    return qualifying


# ---------------------------------------------------------------
# Stage-1 training data
# ---------------------------------------------------------------
# ADDED: download train shards one at a time and stop as soon as
# 5,000 qualifying examples are available.
train_pool = []

for shard_index, url in enumerate(
    TRAIN_PARQUET_URLS
):
    shard_path = (
        PARQUET_CACHE_DIR
        / f"train-{shard_index:05d}.parquet"
    )

    download_file(
        url,
        shard_path,
    )

    shard_records = records_from_parquet(
        shard_path
    )

    existing_train_ids = {record['article_id'] for record in train_pool}    
    existing_train_articles = {record['article'] for record in train_pool}
    shard_records = [
        record for record in shard_records
        if record['article_id'] not in existing_train_ids
        and record['article'] not in existing_train_articles
    ]

    train_pool.extend(shard_records)

    print(
        f"Train shard {shard_index}: "
        f"{len(shard_records)} qualifying; "
        f"pool={len(train_pool)}"
    )

    if (len(train_pool) >= TRAIN_SAMPLE_SIZE):
        break


if len(train_pool) < TRAIN_SAMPLE_SIZE:
    raise RuntimeError(
        f"Only {len(train_pool)} qualifying training records "
        f"were found; {TRAIN_SAMPLE_SIZE} are required."
    )


# Shuffle locally and select the requested number.
random.Random(SEED).shuffle(train_pool)
stage1_train_records = train_pool[:TRAIN_SAMPLE_SIZE]


# ---------------------------------------------------------------
# Stage-1 validation data
# ---------------------------------------------------------------
validation_path = (PARQUET_CACHE_DIR / "validation.parquet")

download_file(VALIDATION_PARQUET_URL, validation_path)

validation_pool = records_from_parquet(validation_path)
selected_train_ids = {record['article_id'] for record in stage1_train_records}
selected_train_articles = {record['article'] for record in stage1_train_records}
validation_pool = [
    record for record in validation_pool
    if record['article_id'] not in selected_train_ids
    and record['article'] not in selected_train_articles
]

if len(validation_pool) < VAL_SAMPLE_SIZE:
    raise RuntimeError(
        f"Only {len(validation_pool)} qualifying validation records "
        f"were found; {VAL_SAMPLE_SIZE} are required."
    )

random.Random(SEED + 1).shuffle(validation_pool)
stage1_val_records = validation_pool[:VAL_SAMPLE_SIZE]
records = stage1_train_records + stage1_val_records

print("\nPrepared Stage-1 training:", len(stage1_train_records))

print("Prepared Stage-1 validation:", len(stage1_val_records))


Loaded 500 Stage-2 IDs to exclude from Stage 1.
Using cached file: data\raw\cnn_dailymail_parquet\train-00000.parquet
Train shard 0: 34814 qualifying; pool=34814
Saved: data\raw\cnn_dailymail_parquet\validation.parquet (33.1 MB)

Prepared Stage-1 training: 5000
Prepared Stage-1 validation: 500


## Save and verify the raw dataset

The selected records are saved locally as JSONL. The training and validation files remain separate so the OPT-2.7B notebook can load them without contacting Hugging Face.


In [3]:
def write_jsonl_atomic(
    path: Path,
    records_to_write: list,
):
    """Write safely, replacing the target only after serialization succeeds."""
    if (
        path.exists()
        and not OVERWRITE
    ):
        raise FileExistsError(
            f"{path} already exists. "
            "Set OVERWRITE=True to replace it."
        )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    try:
        with temporary_path.open(
            "w",
            encoding="utf-8",
            newline="\n",
        ) as handle:
            for record in records_to_write:
                handle.write(
                    json.dumps(
                        record,
                        ensure_ascii=False,
                    )
                    + "\n"
                )

        os.replace(
            temporary_path,
            path,
        )

    finally:
        if temporary_path.exists():
            temporary_path.unlink()


assert (
    len(stage1_train_records)
    == TRAIN_SAMPLE_SIZE
)

assert (
    len(stage1_val_records)
    == VAL_SAMPLE_SIZE
)

for record in records:
    assert set(record) == set(
        RAW_COLUMNS
    )
    assert record[
        "article_id"
    ].strip()
    assert record[
        "article"
    ].strip()
    assert record[
        "summary"
    ].strip()
    assert (
        count_words(
            record["article"]
        )
        == record[
            "article_word_count"
        ]
    )
    assert (
        record[
            "article_word_count"
        ]
        < MAX_ARTICLE_WORDS_EXCLUSIVE
    )


train_ids = {
    record["article_id"]
    for record in stage1_train_records
}

val_ids = {
    record["article_id"]
    for record in stage1_val_records
}

overlap = (
    train_ids
    & val_ids
)

if overlap:
    raise ValueError(
        "Stage-1 train/validation overlap detected. "
        f"Examples: {list(overlap)[:5]}"
    )

stage1_ids = train_ids | val_ids
cross_stage_overlap = stage1_ids & STAGE2_EXCLUDED_IDS

if cross_stage_overlap:
    raise ValueError(
        "Stage-1/Stage-2 overlap detected after replacement. "
        f"Examples: {list(cross_stage_overlap)[:5]}"
    )


write_jsonl_atomic(
    RAW_PATH,
    records,
)

write_jsonl_atomic(
    STAGE1_TRAIN_PATH,
    stage1_train_records,
)

write_jsonl_atomic(
    STAGE1_VAL_PATH,
    stage1_val_records,
)


def read_jsonl(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:
        return [
            json.loads(line)
            for line in handle
            if line.strip()
        ]


saved_train = read_jsonl(
    STAGE1_TRAIN_PATH
)

saved_val = read_jsonl(
    STAGE1_VAL_PATH
)

assert (
    len(saved_train)
    == TRAIN_SAMPLE_SIZE
)

assert (
    len(saved_val)
    == VAL_SAMPLE_SIZE
)

saved_train_ids = {
    record["article_id"]
    for record in saved_train
}

saved_val_ids = {
    record["article_id"]
    for record in saved_val
}

assert saved_train_ids.isdisjoint(
    saved_val_ids
)

all_word_counts = [
    record[
        "article_word_count"
    ]
    for record in (
        saved_train
        + saved_val
    )
]

print(
    f"Saved {len(saved_train)} Stage-1 training records to "
    f"'{STAGE1_TRAIN_PATH}'"
)

print(
    f"Saved {len(saved_val)} Stage-1 validation records to "
    f"'{STAGE1_VAL_PATH}'"
)

print(
    f"Word-count range: "
    f"{min(all_word_counts)} to {max(all_word_counts)}"
)

print(
    "Train/validation overlap: 0"
)

print("Stage-1/Stage-2 overlap: 0")

print(
    "Verification passed."
)


Saved 5000 Stage-1 training records to 'data\processed\stage1_train.jsonl'
Saved 500 Stage-1 validation records to 'data\processed\stage1_validation.jsonl'
Word-count range: 14 to 499
Train/validation overlap: 0
Stage-1/Stage-2 overlap: 0
Verification passed.
